# 01 — You Are the Evaluator

Forget RAGAs. Forget DeepEval. Forget every framework name from the theory phase. For this notebook and the next, you're not writing code that judges anything -- you're reading 20 real answers and deciding, yourself, whether each one is actually good.

This matters more than it sounds like it should. Frameworks and LLM judges are automating a specific human judgment. If you've never made that judgment yourself, you have no way to tell whether the automation is doing it well or just doing it confidently.


In [ ]:
import json
import os

DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)

print(f"Loaded {len(examples)} examples:")
print("  [0-4]   CUAD -- single-fact lookup (Governing Law)")
print("  [5-9]   CUAD -- numeric/date extraction (Cap On Liability)")
print("  [10-14] HotpotQA -- multi-document comparison")
print("  [15-19] SQuAD 2.0 -- unanswerable questions")


## Worked example 1 — the easy, clean case

Start with the one that should be obvious, so you can calibrate what "good" looks like before anything ambiguous shows up.


In [ ]:
ex = examples[0]
print("Question:", ex["question"])
print("Context: ...", ex["context_text"][:250], "...")
print("Gold answer:", ex["gold_answer"])
print("System answer:", ex["system_answer"])


**Judgment:** good. The system answer says "the law of the People's Republic of China; if that does not apply, the United Nations Convention on Contracts for the International Sale of Goods applies" -- that's the gold answer, reworded, not shortened or changed in meaning. Nothing invented, nothing missing. This is what "correct" looks like when it's actually simple.

Notice what you just did, though: you didn't compare strings. `"the law of the People's Republic of China"` and `"It will be governed by the law of the People's Republic of China"` aren't the same characters, but they're the same *claim*. Exact-match scoring would call this wrong. You didn't.


## Worked example 2 — correct, but it needed two facts combined

HotpotQA questions are built to need more than one passage. Watch how the answer has to reach across both halves of the context to be right.


In [ ]:
ex = examples[10]
print("Question:", ex["question"])
print("Context:", ex["context_text"][:400], "...")
print("Gold answer:", ex["gold_answer"])
print("System answer:", ex["system_answer"])


**Judgment:** good. The question asks whether two people share a nationality. The context has to state Scott Derrickson's nationality in one sentence and Ed Wood's in another, and the answer only works if both get pulled in and compared. "Yes" alone (matching gold) would technically score correct under exact-match too -- but the system's actual sentence, "Both Scott Derrickson and Ed Wood are American," is the more useful signal here: it shows *which* two facts got combined, not just that the final yes/no landed right. That distinction -- a right answer for a traceable reason versus a right answer you can't explain -- is exactly what root-cause analysis in the next notebook depends on.


## Worked example 3 — where it gets genuinely interesting

Here's the one that isn't a clean "the system nailed it" or "the system missed it." Read the whole context window before you decide anything.


In [ ]:
ex = examples[8]
print("Question:", ex["question"])
print("Context:")
print(ex["context_text"])
print()
print("Gold answer:", ex["gold_answer"])
print("System answer:", ex["system_answer"])


**Judgment:** bad -- but pay attention to *why*, because the obvious guess is wrong. The gold clause is sitting right there in the middle of the window you just read: *"neither Party shall be responsible to the other Party in respect of any indirect loss or damage caused hereunder."* The system claimed the context contains no information about a cap on liability. It had the answer in front of it and said it didn't.

Before you file this under "the model can't read," look closer at what this clause actually says. It never uses the word "cap," never states a dollar figure, never says anything resembling "liability shall not exceed X." It's an *exclusion* -- indirect and consequential damages are carved out entirely -- which is a different shape of liability limitation than a numeric cap, even though CUAD's own labelers filed it under the same "Cap On Liability" category. A model pattern-matching for "a cap" can reasonably not recognize an exclusion clause as one.

This is worth sitting with, because it's the first real example in this series of a judgment call that doesn't resolve to a one-line verdict. Is it a **generation failure** (missed something explicitly present)? Or is it closer to a **label/task-definition mismatch** (the category "Cap On Liability" is broader than what was asked for)? Both readings are defensible. Write down which one you buy, and why -- you'll want that reasoning again in notebook `02`.


## Worked example 4 — faithful vs. unfaithful, same question

Last worked one: a genuinely unanswerable question, answered two different ways by the exact same model.


In [ ]:
ex = examples[16]
print("Question:", ex["question"])
print("Context:", ex["context_text"][:300], "...")
print("Gold answer:", ex["gold_answer"])
print("Honest system answer:      ", ex["system_answer"])
print("Overconfident system answer:", ex["system_answer_overconfident"])


**Judgment:** the honest version is good -- "the context does not contain an answer to that question" is exactly right, since the passage never says what France is a region of. The overconfident version, "France is a region of Europe," is bad: it's not unreasonable as a fact, but nothing in the given context supports it. That's the entire distinction faithfulness measures, and you just made the call yourself, with no framework involved: **does the context actually say this, or does it just sound like something that could be true?**


## Your turn: the remaining 16

Same four questions, every time: did the answer use the given context? Did it miss anything important? Did it invent something not supported by the context? If the question was unanswerable, did the answer correctly say so, or quietly make something up instead?

Print each one, read it, and fill in your own verdict below. Don't rush past the ones that feel obvious -- example 8 above felt obvious too, right up until you read the clause closely.


In [ ]:
remaining = [i for i in range(20) if i not in (0, 8, 10, 16)]

for i in remaining:
    ex = examples[i]
    print(f"[{i}] ({ex['source']}) {ex['question']}")
    print(f"  Context: {ex['context_text'][:200]}...")
    print(f"  Gold:    {ex['gold_answer'][:150]}")
    print(f"  System:  {ex['system_answer'][:200]}")
    if "system_answer_overconfident" in ex:
        print(f"  Overconfident: {ex['system_answer_overconfident'][:200]}")
    print()


In [ ]:
my_judgments = {
    0: {"verdict": "good", "reason": "Correctly paraphrases the governing-law clause."},
    8: {"verdict": "bad", "reason": "Missed an exclusion-style liability clause present in the context."},
    10: {"verdict": "good", "reason": "Correctly combines both passages' nationality facts."},
    16: {"verdict": "good", "reason": "Honest version correctly declines; overconfident version fabricates a claim."},
    # Fill in the rest after reading the printed output above:
    # 1: {"verdict": "", "reason": ""},
    # 2: {"verdict": "", "reason": ""},
    # ... through 19
}

print(f"Judged so far: {len(my_judgments)}/20")


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Exact-match scoring | Comparing answers as literal strings -- fails the moment wording differs, even when meaning matches |
| Grounded (faithful) answer | An answer that only claims what the context actually supports |
| Label/task-definition mismatch | A "wrong" answer that's actually a reasonable reading of an ambiguous category, not a real failure |

The single most important thing that happened in this notebook wasn't any individual verdict -- it's that you now have a *reason* attached to every judgment, not just a score. "Bad" without a reason is a dead end. "Bad, because the clause is an exclusion not an explicit cap, and the model was pattern-matching for cap language" is something you can actually act on. That distinction is the entire subject of the next notebook.

**Next up:** notebook `02` takes every "bad" verdict you just wrote down and asks the harder question -- not just *that* it's wrong, but specifically *where in the pipeline* it went wrong.

**Exercises:**

- Finish `my_judgments` for all 20 examples before moving on -- this series assumes you have real opinions about all of them, not just the 4 worked ones.
- Go back to example 8. Would you have called it "bad" if the question had been phrased "Does this contract exclude any type of damages?" instead of "Cap On Liability"? What does that tell you about how much a verdict depends on the question's exact wording, not just the answer?
- Pick one CUAD example you marked "good." Try to state, in one sentence, what specifically would have had to change in the system's answer for you to downgrade it to "partially good."
